# Advanced Python for Data Engineering
## Production-Grade Patterns, Design, and Performance

**What you'll learn:**
- Advanced OOP: inheritance, abstract classes, mixins, design patterns
- Generators & iterators for memory-efficient data processing
- Advanced decorators: parameterized, class-based, stacking
- Context managers for resource management
- Concurrency: threading, multiprocessing, concurrent.futures
- Functional programming: functools, itertools, operator
- Dataclasses, Protocols, and modern Python typing
- Production pipeline architecture patterns
- Memory management and performance profiling
- Building reusable data engineering frameworks

**Prerequisites:** 03_python_foundations.ipynb
**Estimated Time:** 8-10 hours

---

> The difference between a junior and senior data engineer is not knowing MORE syntax --
> it's knowing how to structure code for maintainability, testability, and performance.

---

---
# Section 1: Advanced Object-Oriented Programming

## Beyond Basic Classes: Inheritance, Abstract Classes & Design Patterns

Production pipelines use OOP extensively for:
- **Reusable components** (transformers, validators, connectors)
- **Plugin architectures** (add new sources/sinks without changing core)
- **Configuration management** (environments, secrets)
- **State machines** (pipeline status tracking)

In [ ]:
# Abstract Base Classes: Define contracts for pipeline components
from abc import ABC, abstractmethod
from typing import Any, Dict, List, Optional
from datetime import datetime

class DataSource(ABC):
    """Abstract base class for all data sources.
    Every source must implement extract() and validate().
    This is the 'contract' that all sources follow."""

    def __init__(self, name: str, config: Dict[str, Any]):
        self.name = name
        self.config = config
        self.last_extract_time: Optional[datetime] = None
        self._record_count = 0

    @abstractmethod
    def connect(self) -> bool:
        """Establish connection to the source."""
        pass

    @abstractmethod
    def extract(self, **kwargs) -> List[Dict]:
        """Extract data from the source."""
        pass

    @abstractmethod
    def validate(self) -> bool:
        """Validate that the source is healthy."""
        pass

    def get_metrics(self) -> Dict[str, Any]:
        """Common metrics (not abstract -- shared implementation)."""
        return {
            "source": self.name,
            "last_extract": self.last_extract_time,
            "records_extracted": self._record_count
        }


class DatabaseSource(DataSource):
    """Concrete implementation for database sources."""

    def connect(self) -> bool:
        host = self.config.get("host", "localhost")
        print(f"  Connecting to database at {host}...")
        return True

    def extract(self, query: str = "SELECT 1", limit: int = 1000) -> List[Dict]:
        self.last_extract_time = datetime.now()
        # Simulate extraction
        records = [{"id": i, "value": f"row_{i}"} for i in range(min(limit, 10))]
        self._record_count = len(records)
        return records

    def validate(self) -> bool:
        return self.config.get("host") is not None


class APISource(DataSource):
    """Concrete implementation for REST API sources."""

    def connect(self) -> bool:
        url = self.config.get("base_url", "")
        print(f"  Connecting to API at {url}...")
        return True

    def extract(self, endpoint: str = "/data", **kwargs) -> List[Dict]:
        self.last_extract_time = datetime.now()
        records = [{"id": i, "payload": f"api_data_{i}"} for i in range(5)]
        self._record_count = len(records)
        return records

    def validate(self) -> bool:
        return "base_url" in self.config and "api_key" in self.config


# Usage: polymorphism in action
sources: List[DataSource] = [
    DatabaseSource("postgres_orders", {"host": "db.prod.internal", "port": 5432}),
    APISource("stripe_payments", {"base_url": "https://api.stripe.com", "api_key": "sk_live_xxx"}),
]

print("DATA SOURCE ABSTRACTION")
print("=" * 60)
for source in sources:
    source.connect()
    data = source.extract()
    print(f"  {source.name}: extracted {len(data)} records")
    print(f"  Valid: {source.validate()}")
    print(f"  Metrics: {source.get_metrics()}")
    print()

In [ ]:
# Design Pattern: Strategy Pattern for Transformations
from abc import ABC, abstractmethod
from typing import List, Dict

class TransformStrategy(ABC):
    """Strategy interface for data transformations."""

    @abstractmethod
    def transform(self, records: List[Dict]) -> List[Dict]:
        pass

    @property
    @abstractmethod
    def name(self) -> str:
        pass


class CleanNulls(TransformStrategy):
    """Remove records with null required fields."""

    name = "clean_nulls"

    def __init__(self, required_fields: List[str]):
        self.required_fields = required_fields

    def transform(self, records: List[Dict]) -> List[Dict]:
        return [r for r in records if all(r.get(f) is not None for f in self.required_fields)]


class NormalizeCase(TransformStrategy):
    """Normalize string fields to lowercase."""

    name = "normalize_case"

    def __init__(self, fields: List[str]):
        self.fields = fields

    def transform(self, records: List[Dict]) -> List[Dict]:
        for r in records:
            for f in self.fields:
                if isinstance(r.get(f), str):
                    r[f] = r[f].strip().lower()
        return records


class AddAuditFields(TransformStrategy):
    """Add processing metadata."""

    name = "add_audit"

    def transform(self, records: List[Dict]) -> List[Dict]:
        ts = datetime.now().isoformat()
        for r in records:
            r["_processed_at"] = ts
            r["_version"] = 1
        return records


class TransformPipeline:
    """Compose multiple strategies into a pipeline."""

    def __init__(self, strategies: List[TransformStrategy]):
        self.strategies = strategies

    def execute(self, records: List[Dict]) -> List[Dict]:
        result = records
        for strategy in self.strategies:
            before = len(result)
            result = strategy.transform(result)
            after = len(result)
            if before != after:
                print(f"    {strategy.name}: {before} -> {after} records")
        return result


# Usage
raw_data = [
    {"id": 1, "name": "  ALICE  ", "email": "ALICE@GMAIL.COM", "amount": 100},
    {"id": 2, "name": "Bob", "email": None, "amount": 200},
    {"id": 3, "name": "CHARLIE", "email": "charlie@yahoo.com", "amount": None},
    {"id": 4, "name": "  diana ", "email": "DIANA@OUTLOOK.COM", "amount": 150},
]

pipeline = TransformPipeline([
    CleanNulls(required_fields=["email", "amount"]),
    NormalizeCase(fields=["name", "email"]),
    AddAuditFields(),
])

print("STRATEGY PATTERN: Transform Pipeline")
print("=" * 60)
print(f"  Input: {len(raw_data)} records")
result = pipeline.execute(raw_data)
print(f"  Output: {len(result)} records")
for r in result:
    print(f"    {r}")

---
# Section 2: Generators & Iterators

## Processing Billions of Records Without Running Out of Memory

Generators produce values ONE AT A TIME (lazy evaluation).
Instead of loading 10GB into memory, you process row by row.

**Key insight:** A generator is like a pipeline -- data flows through it
without ever being fully materialized in memory.

## When to Use Generators

- Reading large files line by line
- Processing database cursors
- Streaming transformations
- Building data pipelines that compose
- Any time you don't need ALL data in memory at once

In [ ]:
# Generator basics: memory-efficient data processing
import sys

print("GENERATORS: Memory-Efficient Processing")
print("=" * 60)

# List vs Generator memory comparison
list_data = [i * i for i in range(1_000_000)]  # All in memory
gen_data = (i * i for i in range(1_000_000))   # Lazy -- nothing computed yet

print(f"  List memory: {sys.getsizeof(list_data):,} bytes ({sys.getsizeof(list_data) // 1024:,} KB)")
print(f"  Generator memory: {sys.getsizeof(gen_data)} bytes")
print(f"  Ratio: {sys.getsizeof(list_data) // sys.getsizeof(gen_data)}x less memory!")
print()

# Generator function
def read_batches(total_records: int, batch_size: int):
    """Simulate reading records in batches (like a database cursor)."""
    for offset in range(0, total_records, batch_size):
        batch = [{"id": i, "value": i * 10} for i in range(offset, min(offset + batch_size, total_records))]
        yield batch  # Pause here, resume when next() is called

# Process 1M records in 10K batches -- only 10K in memory at a time
total_processed = 0
for batch in read_batches(100_000, 10_000):
    total_processed += len(batch)
    # Process batch here (write to DB, transform, etc.)

print(f"  Processed {total_processed:,} records in batches of 10,000")
print(f"  Max memory: ~10,000 records (not 100,000)")

In [ ]:
# Generator pipelines: compose transformations lazily
from typing import Generator, Dict, Iterator

def read_source(n: int) -> Generator[Dict, None, None]:
    """Simulate reading from a data source."""
    for i in range(n):
        yield {"id": i, "name": f"user_{i}", "amount": (i * 37) % 1000, "status": "active" if i % 3 else "inactive"}

def filter_active(records: Iterator[Dict]) -> Generator[Dict, None, None]:
    """Pass through only active records."""
    for record in records:
        if record["status"] == "active":
            yield record

def enrich(records: Iterator[Dict]) -> Generator[Dict, None, None]:
    """Add computed fields."""
    for record in records:
        record["tier"] = "gold" if record["amount"] > 500 else "silver"
        record["tax"] = round(record["amount"] * 0.08, 2)
        yield record

def batch_writer(records: Iterator[Dict], batch_size: int = 100) -> Generator[list, None, None]:
    """Collect records into batches for bulk writing."""
    batch = []
    for record in records:
        batch.append(record)
        if len(batch) >= batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

# Compose the pipeline (NOTHING executes yet -- it is all lazy!)
source = read_source(1000)
filtered = filter_active(source)
enriched = enrich(filtered)
batches = batch_writer(enriched, batch_size=200)

# NOW process: each record flows through the entire chain
print("GENERATOR PIPELINE (lazy composition)")
print("=" * 60)
total_records = 0
total_batches = 0
for batch in batches:
    total_batches += 1
    total_records += len(batch)

print(f"  Total records processed: {total_records}")
print(f"  Total batches written: {total_batches}")
print(f"  Memory: only 1 batch (~200 records) in memory at any time!")

---
# Section 3: Advanced Decorators

## Decorators That Production Pipelines Actually Use

The foundation covered basic decorators. Here we cover:
- Parameterized decorators (decorators that take arguments)
- Decorators that preserve function metadata
- Class-based decorators
- Stacking multiple decorators
- Real-world patterns: retry, cache, timing, validation

In [ ]:
# Production decorator: retry with exponential backoff
import time
import functools
import random
from typing import Tuple, Type

def retry(
    max_attempts: int = 3,
    delay: float = 1.0,
    backoff: float = 2.0,
    exceptions: Tuple[Type[Exception], ...] = (Exception,),
    on_retry=None
):
    """
    Parameterized retry decorator with exponential backoff.
    Used in EVERY production pipeline for:
    - API calls (network timeouts)
    - Database connections (transient failures)
    - File operations (lock contention)
    """
    def decorator(func):
        @functools.wraps(func)  # Preserves original function metadata
        def wrapper(*args, **kwargs):
            current_delay = delay
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    if attempt == max_attempts:
                        raise
                    if on_retry:
                        on_retry(attempt, e, current_delay)
                    time.sleep(current_delay * 0.01)  # Shortened for demo
                    current_delay *= backoff
        return wrapper
    return decorator


def log_retry(attempt, error, delay):
    print(f"    Attempt {attempt} failed: {error}. Retrying in {delay:.1f}s...")


# Simulated flaky operation
call_counter = {"n": 0}

@retry(max_attempts=4, delay=0.5, backoff=2.0, exceptions=(ConnectionError, TimeoutError), on_retry=log_retry)
def fetch_api_data(endpoint: str) -> dict:
    """Simulates a flaky API call."""
    call_counter["n"] += 1
    if call_counter["n"] <= 2:
        raise ConnectionError(f"Timeout connecting to {endpoint}")
    return {"status": "success", "data": [1, 2, 3]}


print("PARAMETERIZED RETRY DECORATOR")
print("=" * 60)
call_counter["n"] = 0
result = fetch_api_data("/api/orders")
print(f"  Result: {result}")
print(f"  Total attempts: {call_counter['n']}")

In [ ]:
# Production decorator: timing + metrics collection
import time
import functools
from typing import Dict, Any

# Global metrics store
_metrics: Dict[str, list] = {}

def track_performance(metric_name: str = None):
    """Decorator that tracks execution time and stores metrics."""
    def decorator(func):
        name = metric_name or f"{func.__module__}.{func.__qualname__}"

        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            start = time.perf_counter()
            try:
                result = func(*args, **kwargs)
                status = "success"
                return result
            except Exception as e:
                status = "error"
                raise
            finally:
                elapsed = time.perf_counter() - start
                if name not in _metrics:
                    _metrics[name] = []
                _metrics[name].append({"elapsed": elapsed, "status": status})

        return wrapper
    return decorator


def validate_input(**validators):
    """Decorator that validates function arguments before execution."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # Match positional args to parameter names
            import inspect
            sig = inspect.signature(func)
            bound = sig.bind(*args, **kwargs)
            bound.apply_defaults()

            for param_name, validator_fn in validators.items():
                if param_name in bound.arguments:
                    value = bound.arguments[param_name]
                    if not validator_fn(value):
                        raise ValueError(f"Validation failed for '{param_name}': {value}")
            return func(*args, **kwargs)
        return wrapper
    return decorator


# Stack multiple decorators
@track_performance("pipeline.transform_orders")
@validate_input(records=lambda r: isinstance(r, list) and len(r) > 0)
def transform_orders(records: list) -> list:
    """Transform order records with validation and timing."""
    time.sleep(0.01)  # Simulate work
    return [{"id": r["id"], "amount": r["amount"] * 1.08} for r in records]


print("STACKED DECORATORS: Validation + Timing")
print("=" * 60)

# Valid call
data = [{"id": 1, "amount": 100}, {"id": 2, "amount": 200}]
result = transform_orders(data)
print(f"  Valid call result: {result}")
print(f"  Metrics: {_metrics}")

# Invalid call
try:
    transform_orders([])  # Empty list fails validation
except ValueError as e:
    print(f"  Invalid call caught: {e}")

---
# Section 4: Context Managers

## Guaranteed Resource Cleanup

Context managers ensure resources (files, connections, locks) are properly
released even when exceptions occur. Essential for production pipelines.

In [ ]:
# Custom context manager: Database connection pool
from contextlib import contextmanager
from typing import Generator
import time

class ConnectionPool:
    """Simulated database connection pool."""

    def __init__(self, max_connections: int = 5):
        self.max_connections = max_connections
        self.available = list(range(max_connections))
        self.in_use = []

    @contextmanager
    def get_connection(self) -> Generator:
        """Get a connection from the pool, auto-return when done."""
        if not self.available:
            raise RuntimeError("No connections available!")

        conn_id = self.available.pop()
        self.in_use.append(conn_id)
        print(f"    Acquired connection {conn_id} (available: {len(self.available)})")

        try:
            yield conn_id
        finally:
            # ALWAYS returns connection, even if exception occurs
            self.in_use.remove(conn_id)
            self.available.append(conn_id)
            print(f"    Released connection {conn_id} (available: {len(self.available)})")


@contextmanager
def pipeline_stage(stage_name: str):
    """Context manager for pipeline stage tracking."""
    start = time.perf_counter()
    print(f"  [{stage_name}] Starting...")
    try:
        yield
        elapsed = time.perf_counter() - start
        print(f"  [{stage_name}] Completed in {elapsed:.3f}s")
    except Exception as e:
        elapsed = time.perf_counter() - start
        print(f"  [{stage_name}] FAILED after {elapsed:.3f}s: {e}")
        raise


# Usage
print("CONTEXT MANAGERS: Resource Management")
print("=" * 60)

pool = ConnectionPool(max_connections=3)

with pipeline_stage("Extract"):
    with pool.get_connection() as conn:
        print(f"    Using connection {conn} for extraction...")
        time.sleep(0.01)

with pipeline_stage("Transform"):
    time.sleep(0.01)
    print("    Transforming data...")

# Connection is guaranteed to be returned even on error
print("\n  Connection pool status:")
print(f"    Available: {len(pool.available)}, In use: {len(pool.in_use)}")

---
# Section 5: Concurrency & Parallelism

## When Sequential Is Too Slow

Data engineering often involves I/O-bound tasks (API calls, DB queries, file reads)
that benefit hugely from concurrency.

| Pattern | Best For | Python Module |
|---------|----------|---------------|
| **Threading** | I/O-bound (API calls, DB) | `threading`, `concurrent.futures` |
| **Multiprocessing** | CPU-bound (transformations) | `multiprocessing`, `concurrent.futures` |
| **Async** | Many concurrent I/O operations | `asyncio` (covered in notebook 30) |

## Rule of Thumb
- 10 API calls sequentially = 10 seconds
- 10 API calls concurrently = ~1 second

In [ ]:
# concurrent.futures: Thread pool for I/O-bound work
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
import time

def fetch_partition(partition_id: int) -> dict:
    """Simulate fetching data from a partition (I/O-bound)."""
    time.sleep(0.1)  # Simulate network/disk I/O
    return {"partition": partition_id, "rows": partition_id * 1000, "status": "done"}

print("CONCURRENT EXECUTION")
print("=" * 60)

# Sequential (slow)
start = time.perf_counter()
sequential_results = [fetch_partition(i) for i in range(10)]
seq_time = time.perf_counter() - start
print(f"  Sequential (10 partitions): {seq_time:.2f}s")

# Parallel with ThreadPool (fast for I/O)
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {executor.submit(fetch_partition, i): i for i in range(10)}
    parallel_results = []
    for future in as_completed(futures):
        parallel_results.append(future.result())
par_time = time.perf_counter() - start
print(f"  Parallel (5 threads):       {par_time:.2f}s")
print(f"  Speedup: {seq_time / par_time:.1f}x faster!")

# ProcessPool for CPU-bound work
def heavy_transform(data: list) -> list:
    """Simulate CPU-heavy transformation."""
    return [x ** 2 for x in data]

start = time.perf_counter()
chunks = [list(range(i * 10000, (i + 1) * 10000)) for i in range(4)]
with ProcessPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(heavy_transform, chunks))
proc_time = time.perf_counter() - start
total_items = sum(len(r) for r in results)
print(f"\n  ProcessPool (4 workers, {total_items:,} items): {proc_time:.3f}s")

---
# Section 6: Functional Programming Patterns

## functools, itertools, and operator

Functional patterns are everywhere in PySpark (map, filter, reduce).
Understanding them in pure Python makes you better at Spark too.

In [ ]:
# functools: partial, reduce, lru_cache
import functools
import itertools
import operator

print("FUNCTOOLS PATTERNS")
print("=" * 60)

# partial: Pre-fill arguments for reusable functions
def apply_tax(amount: float, rate: float) -> float:
    return round(amount * (1 + rate), 2)

apply_us_tax = functools.partial(apply_tax, rate=0.08)
apply_uk_vat = functools.partial(apply_tax, rate=0.20)

print("  partial -- pre-configured functions:")
print(f"    US tax on $100: ${apply_us_tax(100)}")
print(f"    UK VAT on $100: ${apply_uk_vat(100)}")

# reduce: Aggregate a sequence into a single value
orders = [150.0, 230.0, 89.0, 410.0, 67.0]
total = functools.reduce(operator.add, orders)
print(f"\n  reduce -- sum of orders: ${total}")

# Composing functions with reduce
def compose(*functions):
    """Compose functions: compose(f, g, h)(x) = f(g(h(x)))"""
    return functools.reduce(lambda f, g: lambda x: f(g(x)), functions)

clean = compose(str.strip, str.lower, lambda s: s.replace("  ", " "))
print(f"\n  compose -- clean string: '{clean('  HELLO   WORLD  ')}'")

# lru_cache: Memoize expensive computations
@functools.lru_cache(maxsize=128)
def expensive_lookup(customer_id: int) -> dict:
    """Simulate expensive DB lookup (cached after first call)."""
    time.sleep(0.01)  # Simulate latency
    return {"id": customer_id, "segment": "premium" if customer_id % 2 else "standard"}

import time
start = time.perf_counter()
for _ in range(100):
    expensive_lookup(42)  # Only slow the first time
elapsed = time.perf_counter() - start
print(f"\n  lru_cache -- 100 lookups of same key: {elapsed:.4f}s (cached after first)")
print(f"    Cache info: {expensive_lookup.cache_info()}")

In [ ]:
# itertools: Efficient iteration patterns
import itertools

print("\nITERTOOLS PATTERNS")
print("=" * 60)

# chain: Flatten multiple iterables (like UNION ALL)
source_a = [{"id": 1}, {"id": 2}]
source_b = [{"id": 3}, {"id": 4}]
source_c = [{"id": 5}]

combined = list(itertools.chain(source_a, source_b, source_c))
print(f"  chain (UNION ALL): {len(combined)} records from 3 sources")

# groupby: Group sorted data (like GROUP BY)
records = [
    {"dept": "engineering", "name": "Alice", "salary": 120000},
    {"dept": "engineering", "name": "Bob", "salary": 110000},
    {"dept": "sales", "name": "Charlie", "salary": 95000},
    {"dept": "sales", "name": "Diana", "salary": 105000},
    {"dept": "sales", "name": "Eve", "salary": 90000},
]
# MUST be sorted by key first!
print("\n  groupby (GROUP BY equivalent):")
for dept, group in itertools.groupby(records, key=lambda r: r["dept"]):
    members = list(group)
    avg_salary = sum(m["salary"] for m in members) / len(members)
    print(f"    {dept}: {len(members)} people, avg ${avg_salary:,.0f}")

# islice: Take first N without loading all (like LIMIT)
big_generator = (x for x in range(1_000_000))
first_5 = list(itertools.islice(big_generator, 5))
print(f"\n  islice (LIMIT 5 from 1M): {first_5}")

# batched (Python 3.12+) or manual chunking
def batched(iterable, n):
    """Yield successive n-sized chunks."""
    it = iter(iterable)
    while batch := list(itertools.islice(it, n)):
        yield batch

data = list(range(23))
chunks = list(batched(data, 5))
print(f"\n  batched (chunk 23 items into 5s): {[len(c) for c in chunks]} sizes")

---
# Section 7: Modern Python: Dataclasses, Protocols & Typing

## Writing Self-Documenting, IDE-Friendly Code

Dataclasses eliminate boilerplate. Protocols define interfaces without inheritance.
Type hints catch bugs before runtime.

In [ ]:
# Dataclasses: Clean data containers
from dataclasses import dataclass, field, asdict
from typing import Optional, Protocol
from enum import Enum
from datetime import datetime

class PipelineStatus(Enum):
    PENDING = "pending"
    RUNNING = "running"
    SUCCESS = "success"
    FAILED = "failed"

@dataclass
class PipelineConfig:
    """Configuration for a data pipeline."""
    name: str
    source_table: str
    target_table: str
    batch_size: int = 10000
    max_retries: int = 3
    timeout_seconds: int = 3600
    enabled: bool = True
    tags: list = field(default_factory=list)

    @property
    def fully_qualified_source(self) -> str:
        return f"techmart_dw.{self.source_table}"

    @property
    def fully_qualified_target(self) -> str:
        return f"gold.{self.target_table}"


@dataclass
class PipelineResult:
    """Result of a pipeline execution."""
    config: PipelineConfig
    status: PipelineStatus
    records_processed: int = 0
    records_failed: int = 0
    start_time: Optional[datetime] = None
    end_time: Optional[datetime] = None
    error_message: Optional[str] = None

    @property
    def duration_seconds(self) -> Optional[float]:
        if self.start_time and self.end_time:
            return (self.end_time - self.start_time).total_seconds()
        return None

    @property
    def success_rate(self) -> float:
        total = self.records_processed + self.records_failed
        return self.records_processed / total if total > 0 else 0.0


# Usage
print("DATACLASSES: Clean Pipeline Configuration")
print("=" * 60)

config = PipelineConfig(
    name="daily_orders_gold",
    source_table="orders",
    target_table="orders_summary",
    batch_size=50000,
    tags=["daily", "critical"]
)
print(f"  Config: {config.name}")
print(f"  Source: {config.fully_qualified_source}")
print(f"  Target: {config.fully_qualified_target}")
print(f"  As dict: {asdict(config)}")

result = PipelineResult(
    config=config,
    status=PipelineStatus.SUCCESS,
    records_processed=45000,
    records_failed=12,
    start_time=datetime(2024, 6, 1, 10, 0, 0),
    end_time=datetime(2024, 6, 1, 10, 3, 30),
)
print(f"\n  Result: {result.status.value}")
print(f"  Duration: {result.duration_seconds}s")
print(f"  Success rate: {result.success_rate:.4f}")

In [ ]:
# Protocol: Structural typing (duck typing with type checking)
from typing import Protocol, runtime_checkable

@runtime_checkable
class Writable(Protocol):
    """Any object that can write records. No inheritance needed!"""
    def write(self, records: list) -> int: ...
    def flush(self) -> None: ...

class DeltaWriter:
    """Writes to Delta Lake."""
    def __init__(self, table: str):
        self.table = table
        self.buffer = []

    def write(self, records: list) -> int:
        self.buffer.extend(records)
        return len(records)

    def flush(self) -> None:
        print(f"    Flushing {len(self.buffer)} records to Delta: {self.table}")
        self.buffer = []

class ConsoleWriter:
    """Writes to console (for testing)."""
    def write(self, records: list) -> int:
        for r in records[:3]:
            print(f"    >> {r}")
        return len(records)

    def flush(self) -> None:
        print("    (console flush -- no-op)")


def process_and_write(data: list, writer: Writable) -> int:
    """Works with ANY object that has write() and flush() methods."""
    count = writer.write(data)
    writer.flush()
    return count


print("\nPROTOCOLS: Structural Typing")
print("=" * 60)
data = [{"id": 1}, {"id": 2}, {"id": 3}]

# Both satisfy the Writable protocol without inheriting from it
for writer in [DeltaWriter("gold.orders"), ConsoleWriter()]:
    print(f"  {type(writer).__name__} is Writable: {isinstance(writer, Writable)}")
    process_and_write(data, writer)
    print()

---
# Section 8: Production Pipeline Architecture

## Putting It All Together: A Real Pipeline Framework

This section combines everything into a production-ready pattern that
you'd see at companies like Netflix, Airbnb, or Databricks.

In [ ]:
# Complete production pipeline framework
from dataclasses import dataclass, field
from typing import List, Dict, Any, Callable, Optional
from abc import ABC, abstractmethod
from datetime import datetime
from enum import Enum
import functools
import time

class StageStatus(Enum):
    PENDING = "pending"
    RUNNING = "running"
    SUCCESS = "success"
    FAILED = "failed"
    SKIPPED = "skipped"

@dataclass
class StageResult:
    name: str
    status: StageStatus
    records_in: int = 0
    records_out: int = 0
    duration_ms: float = 0
    error: Optional[str] = None

class Pipeline:
    """
    Production pipeline with:
    - Ordered stage execution
    - Error isolation (one stage fails, others can continue or abort)
    - Metrics collection
    - Dry-run capability
    """

    def __init__(self, name: str, fail_fast: bool = True):
        self.name = name
        self.fail_fast = fail_fast
        self.stages: List[tuple] = []  # (name, function)
        self.results: List[StageResult] = []

    def add_stage(self, name: str, func: Callable):
        """Register a pipeline stage."""
        self.stages.append((name, func))
        return self  # Enable chaining

    def run(self, data: Any = None) -> Dict[str, Any]:
        """Execute all stages sequentially."""
        print(f"  Pipeline '{self.name}' starting ({len(self.stages)} stages)")
        print(f"  {'='*50}")

        current_data = data
        for stage_name, stage_func in self.stages:
            start = time.perf_counter()
            try:
                input_size = len(current_data) if isinstance(current_data, list) else 0
                current_data = stage_func(current_data)
                output_size = len(current_data) if isinstance(current_data, list) else 0
                elapsed = (time.perf_counter() - start) * 1000

                result = StageResult(stage_name, StageStatus.SUCCESS, input_size, output_size, elapsed)
                self.results.append(result)
                print(f"    [{stage_name}] {input_size} -> {output_size} records ({elapsed:.1f}ms)")

            except Exception as e:
                elapsed = (time.perf_counter() - start) * 1000
                result = StageResult(stage_name, StageStatus.FAILED, error=str(e), duration_ms=elapsed)
                self.results.append(result)
                print(f"    [{stage_name}] FAILED: {e}")
                if self.fail_fast:
                    break

        # Summary
        succeeded = sum(1 for r in self.results if r.status == StageStatus.SUCCESS)
        failed = sum(1 for r in self.results if r.status == StageStatus.FAILED)
        total_ms = sum(r.duration_ms for r in self.results)
        print(f"  {'='*50}")
        print(f"  Completed: {succeeded} succeeded, {failed} failed, {total_ms:.1f}ms total")

        return {"data": current_data, "results": self.results}


# Define pipeline stages as simple functions
def extract(data):
    """Simulate extracting raw data."""
    return [{"id": i, "name": f"User {i}", "amount": (i * 37) % 500 + 10, "email": f"user{i}@test.com" if i % 5 else None}
            for i in range(1000)]

def validate(data):
    """Remove invalid records."""
    return [r for r in data if r.get("email") and r["amount"] > 0]

def transform(data):
    """Enrich and transform."""
    for r in data:
        r["amount_with_tax"] = round(r["amount"] * 1.08, 2)
        r["tier"] = "gold" if r["amount"] > 300 else "silver"
    return data

def aggregate(data):
    """Compute summary metrics."""
    from collections import Counter
    tiers = Counter(r["tier"] for r in data)
    return data  # Pass through (side effect: could write summary)


# Build and run
print("PRODUCTION PIPELINE FRAMEWORK")
print("=" * 60)

pipeline = Pipeline("daily_user_etl", fail_fast=True)
pipeline.add_stage("extract", extract)
pipeline.add_stage("validate", validate)
pipeline.add_stage("transform", transform)
pipeline.add_stage("aggregate", aggregate)

output = pipeline.run()
print(f"\n  Final output: {len(output['data'])} records")

---
# Section 9: Your Turn -- Practice Challenges

In [ ]:
# ============================================
# YOUR TURN: Build a Configurable ETL Framework
# ============================================
# Create a class `ConfigurableETL` that:
# 1. Takes a YAML-like config dict with source, transformations, target
# 2. Uses the Strategy pattern for transformations
# 3. Has a dry_run mode that shows what WOULD happen
# 4. Collects metrics (records in/out per stage, timing)
# 5. Supports retry on the extract step
#
# Config example:
# config = {
#     "name": "orders_pipeline",
#     "source": {"type": "table", "name": "orders"},
#     "transformations": ["filter_completed", "add_tax", "categorize"],
#     "target": {"type": "table", "name": "orders_gold"},
#     "retry": {"max_attempts": 3, "delay": 1}
# }
#
# Write your implementation below:

# (Implement here)
print("Challenge: Build a ConfigurableETL class")
print("Use the patterns from this notebook: ABC, Strategy, decorators, context managers")


---
# Summary

## Advanced Python Patterns for DE -- Cheat Sheet

| Pattern | Use Case | Key Benefit |
|---------|----------|-------------|
| **Abstract Base Classes** | Enforce interface contracts | Type safety, documentation |
| **Strategy Pattern** | Swappable algorithms | Open/closed principle |
| **Generators** | Large data processing | Constant memory usage |
| **Decorators** | Cross-cutting concerns | DRY (retry, timing, validation) |
| **Context Managers** | Resource management | Guaranteed cleanup |
| **ThreadPool** | I/O-bound parallelism | 5-10x speedup for API/DB calls |
| **ProcessPool** | CPU-bound parallelism | Use all cores for transforms |
| **functools.partial** | Pre-configured functions | Reusable, composable |
| **itertools** | Efficient iteration | Memory-efficient pipelines |
| **Dataclasses** | Data containers | Less boilerplate, type hints |
| **Protocols** | Structural typing | Flexible interfaces |
| **Pipeline class** | Stage orchestration | Metrics, error isolation |

## What Senior DEs Do Differently

1. **Design before code** -- Draw the architecture, then implement
2. **Compose small functions** -- Each does one thing well
3. **Handle failures gracefully** -- Retry transients, DLQ permanents
4. **Measure everything** -- Timing, record counts, error rates
5. **Make it testable** -- Dependency injection, no hardcoded values
6. **Think about memory** -- Generators for large data, streaming
7. **Use type hints** -- Catch bugs before runtime

---
*Advanced Python for Data Engineering -- Production-grade patterns*

In [ ]:
print("KEY TAKEAWAYS")
print("=" * 50)
print()
print("1. ABC + Strategy = extensible pipeline components")
print("2. Generators = process billions without memory issues")
print("3. Decorators = reusable cross-cutting concerns (retry, timing)")
print("4. Context managers = guaranteed resource cleanup")
print("5. ThreadPool for I/O, ProcessPool for CPU")
print("6. Dataclasses + Protocols = modern, type-safe Python")
print("7. Pipeline class = production orchestration pattern")
print()
print("These patterns appear in every production DE system:")
print("  Netflix (data pipelines), Airbnb (Airflow DAGs),")
print("  Uber (real-time processing), Databricks (SDK internals)")